In [2]:
import pathlib
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from prepare import process_regressor, model_performance_metrics,draw_scatter
from data_prune_function import get_scores,get_data,prune_rd,return_model
random_state =352
test_size = 0.1

model = {}
for modelname in ['CB','XG','GB']:
    model[modelname] = return_model(modelname,random_state=random_state)

target = 'NO(%)'
df,X,y,X_train_val,X_test,y_train_val,y_test = get_data(target = target,random_state=random_state, test_size=test_size,standardized=False)
mad = (y - y.mean()).abs().mean()
std = y.std()
print('')
print('-----------------')
print(f'{target} {y.shape[0]} {mad:.3f} {std:.3f} ')
print('-----------------')
target = 'NO(%)'
a = 'CB'
b = 'XG'
c = 'GB'


-----------------
NO(%) 242 14.156 17.853 
-----------------


In [3]:
train_pred, test_pred = process_regressor(model[a],X_train_val,X_test,y_train_val,y_test)
model_performance_metrics('Cat Train', y_train_val, train_pred)
model_performance_metrics('Cat Test', y_test, test_pred)

cv train R2: 0.685 (+/- 0.094) 

Cat Train mse: 0.00019 | rmse: 0.01389 | mae: 0.01068 | mape: 2.69917 | smape: 2.67146 | r2: 0.99380

Cat Test mse: 0.00383 | rmse: 0.06187 | mae: 0.04682 | mape: 14.87899 | smape: 13.21868 | r2: 0.89445



0.8944527835183073

In [4]:
folder = f'{target}/{target}_{a}_guiding_pruning'
pathlib.Path(f"./{folder}").mkdir(parents=True, exist_ok=True)
file_out_a = f'{folder}/all_dat.pkl'

In [5]:
maes_a, rmse_a, r2_a = get_scores(model[a], X_train_val, y_train_val, X_test, y_test)

Test scores: MAE=4.682, RMSE=6.187, R2=0.894
--- 0.6730453968048096 seconds ---



In [6]:
size_old_val_a, ids_a, test_scores_a, val_scores_a,test_fit_a,remove_a = prune_rd(
        model[a], X_train_val, y_train_val, X_test, y_test,max_iter=80,
        min_drop= 2,threshold=maes_a / 2,train_size= 0.9, file_out=file_out_a)

Iteration 0:
丢弃训练词条数量 : 4 
本次循环中 old_val 数量: 0 (ratio = 0.000)
本次循环中新训练集大小，占比: 217  (ratio = 1.000)
--- 0.6522853374481201 seconds ---
修剪模型：
Test scores: maes=4.616, rmse=5.952, r2=0.902
--- 0.6862719058990479 seconds ---
Iteration 1:
丢弃训练词条数量 : 4 
本次循环中 old_val 数量: 4 (ratio = 0.018)
本次循环中新训练集大小，占比: 213  (ratio = 0.982)
--- 0.6465258598327637 seconds ---
修剪模型：
Test scores: maes=4.761, rmse=6.143, r2=0.896
Val scores: maes=0.916, rmse=1.228, r2=0.996
--- 0.681037187576294 seconds ---
Iteration 2:
丢弃训练词条数量 : 4 
本次循环中 old_val 数量: 8 (ratio = 0.037)
本次循环中新训练集大小，占比: 209  (ratio = 0.963)
--- 0.651334285736084 seconds ---
修剪模型：
Test scores: maes=4.604, rmse=6.107, r2=0.897
Val scores: maes=1.305, rmse=1.535, r2=0.989
--- 0.683368444442749 seconds ---
Iteration 3:
丢弃训练词条数量 : 4 
本次循环中 old_val 数量: 12 (ratio = 0.055)
本次循环中新训练集大小，占比: 205  (ratio = 0.945)
--- 0.6524608135223389 seconds ---
修剪模型：
Test scores: maes=4.724, rmse=6.249, r2=0.892
Val scores: maes=1.931, rmse=2.388, r2=0.982
--- 0.69600915

In [7]:
X_train_val.index
index_list = X_train_val.index.tolist()
df_a_100 = df.loc[index_list]
df_a_90 = df.loc[remove_a[6]]
df_a_70 = df.loc[remove_a[20]]
df_a_50 = df.loc[remove_a[40]]
df_a_30 = df.loc[remove_a[62]]

In [8]:
a = df.drop(df_a_100.index)
a

,Urea,Melamine,Dcda,PM(g),Time(h),Heat(℃),HR(℃/min),hata,modefiy,dope,ratio(%),load(g),area(cm2),NO(ppm),rate(mL/min),Xe,Wu,LED,Intensity(W),BET(m2/g),Eg(eV),NO2(%),NO(%)
8,0,1,0,10.000000,4.0,500,5.000000,0,0,1,1.80,0.08,50.000000,12.00,200.000000,0,0,1,30,17.082199,2.660000,80.500000,91.40
11,1,0,0,20.000000,2.0,600,5.000000,1,0,0,0.50,0.02,26.420000,2.20,1700.000000,1,0,0,300,52.277448,2.699848,39.610000,31.90
14,0,1,0,10.050334,2.0,550,5.000000,0,1,0,33.33,0.30,63.620000,1.00,1000.000000,0,1,0,150,17.229625,2.758549,7.130000,27.69
17,0,1,0,9.284932,2.0,550,5.000000,0,1,0,100.00,0.30,63.620000,1.00,1000.000000,0,1,0,150,7.730200,2.670000,2.380000,39.72
50,0,0,1,20.000000,2.0,550,10.000000,1,0,0,5.26,0.20,103.870000,0.60,1000.000000,0,0,1,15,41.000000,2.595804,7.840000,59.73
56,1,0,0,7.610961,2.0,500,5.000000,1,0,0,11.11,0.10,80.000000,0.60,1200.000000,1,0,0,300,41.464296,2.745978,30.699209,38.00
58,1,0,0,7.849360,2.0,500,5.000000,1,0,0,42.86,0.10,80.000000,0.60,1200.000000,1,0,0,300,57.805225,2.714887,27.957657,57.00
77,1,0,0,10.000000,2.0,550,15.000000,0,0,1,10.00,0.20,113.100000,0.60,2400.000000,0,0,1,30,42.282305,2.574359,37.276045,46.10
84,1,0,0,10.000000,2.0,550,15.000000,1,0,0,100.00,0.20,113.100000,0.60,2415.000000,0,1,0,150,36.000000,2.682864,16.450716,31.40
85,1,0,0,10.000000,2.0,550,15.000000,1,0,0,200.00,0.20,113.100000,0.60,2415.000000,0,1,0,150,15.000000,2.557819,25.360739,18.20


In [9]:
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import KernelDensity
def information_density(X, metric='cosine'):
    """
    X : ndarray, shape (n_samples, n_features)
    返回每个样本的信息密度 D (值越大表示越独特)
    """
    if metric == 'cosine':
        sim_matrix = cosine_similarity(X)  # 全相似度矩阵
    else:
        # 可选欧氏距离转相似度: sim = np.exp(-euclidean_distances(X))
        raise NotImplementedError
    
    # 将对角线（自身相似度=1）设为 -inf，避免取到自身
    np.fill_diagonal(sim_matrix, -np.inf)
    max_sim = np.max(sim_matrix, axis=1)
    # 若最大相似度为 -inf（仅一个样本），则信息密度 = 1
    max_sim = np.where(max_sim == -np.inf, 0, max_sim)
    D = 1 - max_sim
    return D  # shape (n_samples,)

def coverage_integrity(X, bandwidth=1.0):
    """
    基于 KDE 计算每个样本的覆盖完整性 V
    值越大表示该样本位于稀疏区域，对覆盖整体分布贡献越大
    """
    kde = KernelDensity(bandwidth=bandwidth, metric='euclidean')
    kde.fit(X)
    # 计算每个样本的对数密度
    log_density = kde.score_samples(X)
    density = np.exp(log_density)
    
    # 归一化到 [0,1]
    min_d, max_d = density.min(), density.max()
    if max_d > min_d:
        norm_density = (density - min_d) / (max_d - min_d)
    else:
        norm_density = np.zeros_like(density)
    
    # 覆盖完整性 = 1 - 归一化密度 (低密度区域得分高)
    V = 1 - norm_density
    return V

def complementarity(x = None,y = None,  estimator=None):
    """
    X, y: 全部数据
    返回每个样本的互补性 C (值越大表示对性能贡献越大)
    """
    if estimator is None:
        estimator = model[a]
    
    # 固定验证集，用于公平评估性能变化
    d = df.drop(x.index)
    X_test = d.drop(columns = ['NO(%)'])
    y_test = d['NO(%)']

    X_train = x
    y_train = y
    # 基准模型：在全部训练集上训练，在验证集上评估 R²
    estimator.fit(X_train, y_train)
    y_pred_base = estimator.predict(X_test)
    r2_base = r2_score(y_test, y_pred_base)
    
    C_scores = np.zeros(X_train.shape[0])
    # 对每个训练样本进行留一评估
    for i in range(X_train.shape[0]):
        X_loo = np.delete(X_train, i, axis=0)
        y_loo = np.delete(y_train, i)
        estimator.fit(X_loo, y_loo)
        y_pred_loo = estimator.predict(X_test)
        r2_loo = r2_score(y_test, y_pred_loo)
        C_scores[i] = r2_base - r2_loo   # 正数表示移除后性能下降
    
    # 对于原始未进入训练集的样本（测试集），互补性可定义为 0 或单独处理
    # 此处简化：仅返回训练样本的互补性，测试样本补 0
    full_C = np.zeros(X.shape[0])
    # 这里假设 X 的整体顺序和 train_test_split 的索引对应，实际需映射
    # 更严谨的做法：记录 train_index, 然后赋值
    return C_scores  # 仅训练样本的 C

In [10]:
a = 'CB'

D_100 = information_density(df_a_100, metric='cosine')
V_100 = coverage_integrity(df_a_100, bandwidth=1)
C_100 = complementarity(x = df_a_100.drop(columns = ['NO(%)']),y = df_a_100['NO(%)'], estimator=None)

In [11]:
D_90 = information_density(df_a_90, metric='cosine')
V_90 = coverage_integrity(df_a_90, bandwidth=1)
C_90 = complementarity(x = df_a_90.drop(columns = ['NO(%)']),y = df_a_90['NO(%)'], estimator=None)

In [12]:
D_70 = information_density(df_a_70, metric='cosine')
V_70 = coverage_integrity(df_a_70, bandwidth=1)
C_70 = complementarity(x = df_a_70.drop(columns = ['NO(%)']),y = df_a_70['NO(%)'], estimator=None)

In [13]:
D_50 = information_density(df_a_50, metric='cosine')
V_50 = coverage_integrity(df_a_50, bandwidth=1)
C_50 = complementarity(x = df_a_50.drop(columns = ['NO(%)']),y = df_a_50['NO(%)'], estimator=None)

In [14]:
D_30 = information_density(df_a_30, metric='cosine')
V_30 = coverage_integrity(df_a_30, bandwidth=1)
C_30 = complementarity(x = df_a_30.drop(columns = ['NO(%)']),y = df_a_30['NO(%)'], estimator=None)

In [15]:
print(np.mean(D_30))
print(np.mean(D_50))
print(np.mean(D_70))
print(np.mean(D_90))
print(np.mean(D_100))
print(np.mean(C_100))
print(np.mean(C_90))
print(np.mean(C_70))
print(np.mean(C_50))
print(np.mean(C_30))
print(np.mean(V_100))
print(np.mean(V_90))
print(np.mean(V_70))
print(np.mean(V_50))
print(np.mean(V_30))

0.005635646116624699
0.0030259424731806923
0.0022194085633726835
0.0017344776044383638
0.0015545478475291578
-0.006971997229485386
-0.0066092359387975716
0.0034263979826350795
0.00461267372562968
0.012215185121372412
0.9859177658581273
0.989341563583548
0.9866927586932445
0.9683572864580672
0.9692305707386636


In [16]:
def normalize(x):
    return (x-x.min())/(x.max()-x.min()+1e-12)

D100 = normalize(D_100)
C100 = normalize(C_100)
V100 = normalize(V_100)
D90 = normalize(D_90)
C90 = normalize(C_90)
V90 = normalize(V_90)
D70 = normalize(D_70)
C70 = normalize(C_70)
V70 = normalize(V_70)
D50 = normalize(D_50)
C50 = normalize(C_50)
V50 = normalize(V_50)
D30 = normalize(D_30)
C30 = normalize(C_30)
V30 = normalize(V_30)

In [17]:
print(np.mean(D30))
print(np.mean(D50))
print(np.mean(D70))
print(np.mean(D90))
print(np.mean(D100))
print(np.mean(C100))
print(np.mean(C90))
print(np.mean(C70))
print(np.mean(C50))
print(np.mean(C30))

0.018920413247398064
0.010622089007520243
0.007796216132355369
0.0060926021144577435
0.005461166999230701
0.3060344789828491
0.2760910263974903
0.36615423685479664
0.23063995586394023
0.31873680160801554


In [18]:
DRI_sample_100 = (D100 + C100 + V100)/3
DRI_dataset_100 = np.mean(DRI_sample_100)
DIR_sample_90 = (D90 + C90 + V90)/3
DIR_dataset_90 = np.mean(DIR_sample_90)
DIR_sample_70 = (D70 + C70 + V70)/3
DIR_dataset_70 =np.mean(DIR_sample_70)
DIR_sample_50 = (D50 + C50 + V50)/3
DIR_dataset_50 = np.mean(DIR_sample_50)
DIR_sample_30 = (D30 + C30 + V30)/3
DIR_dataset_30 = np.mean(DIR_sample_30)

In [19]:
DRI_sample_100 = (    D100**0.1    *    C100**0.9    *  100)
DRI_dataset_100 = np.mean(DRI_sample_100)
DIR_sample_90 = (    D90**0.1  *    C90**0.9    *    95)
DIR_dataset_90 = np.mean(DIR_sample_90)
DRI_sample_70 = ( D70**0.1 * C70**0.9 * 90)
DRI_dataset_70= np.mean(DRI_sample_70)
DRI_sample_50 = ( D50**0.1 * C50**0.9 * 70)
DIR_dataset_50 = np.mean(DRI_sample_50)
DRI_sample_30 = ( D30**0.1 * C30**0.9 * 50)
DIR_dataset_30 = np.mean(DRI_sample_30)

In [20]:
DRI_dataset_100

np.float64(14.451914301581361)

In [21]:
DIR_dataset_90

np.float64(12.674888307225645)

In [22]:
DRI_dataset_70

np.float64(16.27337519254872)

In [23]:
DIR_dataset_50

np.float64(8.772467254931447)

In [24]:
DIR_dataset_30

np.float64(8.853161170494237)

In [25]:
def kde_coverage(
    X_subset,
    X_full,
    bandwidth=1.0
):

    kde_full = KernelDensity(
        bandwidth=bandwidth
    ).fit(X_full)

    kde_sub = KernelDensity(
        bandwidth=bandwidth
    ).fit(X_subset)

    p_full = np.exp(
        kde_full.score_samples(X_full)
    )

    p_sub = np.exp(
        kde_sub.score_samples(X_full)
    )

    coverage = 1 - (
        np.mean(
            np.abs(
                p_full - p_sub
            )
        )
        /
        np.mean(p_full)
    )

    return max(
        coverage,
        0
    )

In [26]:
Coverage_90 = kde_coverage(df_a_90, df_a_100, bandwidth=10)
Coverage_90

np.float64(0.811873423653771)

In [27]:
Coverage_70 = kde_coverage(df_a_70, df_a_100, bandwidth=10)
Coverage_70

np.float64(0.5476706611592148)

In [28]:
Coverage_50 = kde_coverage(df_a_50, df_a_100, bandwidth=10)
Coverage_50

np.float64(0.32200311996277564)

In [29]:
Coverage_30 = kde_coverage(df_a_30, df_a_100, bandwidth=10)
Coverage_30

np.float64(0.03631919026303376)

In [30]:
DRI_sample_100 = (    D100**0.3    *    C100**0.4    *  1**0.3)
DRI_dataset_100 = np.mean(DRI_sample_100)
print(DRI_dataset_100)
DIR_sample_90 = (    D90**0.3  *    C90**0.4    *    Coverage_90**0.3)
DIR_dataset_90 = np.mean(DIR_sample_90)
print(DIR_dataset_90)
DRI_sample_70 = ( D70**0.3 * C70**0.4 * Coverage_70**0.3)
DRI_dataset_70= np.mean(DRI_sample_70)
print(DRI_dataset_70)
DRI_sample_50 = ( D50**0.3 * C50**0.4 * Coverage_50**0.3)
DIR_dataset_50 = np.mean(DRI_sample_50)
print(DIR_dataset_50)
DRI_sample_30 = ( D30**0.3 * C30**0.4 * Coverage_30**0.3)
DIR_dataset_30 = np.mean(DRI_sample_30)
print(DIR_dataset_30)

0.05386961014953023
0.050444816780437274
0.057106315141262846
0.045669065742262774
0.03219026910754406


In [33]:
DIR_100 = DRI_dataset_100/DRI_dataset_100
print(DIR_100)
DIR_90 = DIR_dataset_90/DRI_dataset_100
print(DIR_90)
DIR_70 = DRI_dataset_70/DRI_dataset_100
print(DIR_70)
DIR_50 = DIR_dataset_50/DRI_dataset_100
print(DIR_50)
DIR_30 = DIR_dataset_30/DRI_dataset_100
print(DIR_30)

1.0
0.9364243891948266
1.0600840619181806
0.8477704890660144
0.5975589765396655


In [202]:
nn_coverage(df_a_70,df_a_100)

np.float64(0.695852534562212)

In [203]:
nn_coverage(df_a_50,df_a_100)

np.float64(0.5023041474654378)

In [204]:
nn_coverage(df_a_30,df_a_100)

np.float64(0.2995391705069124)